# 17 | LangChain：为什么 Agent 需要Middleware

Agent 一旦能自己调用模型、选择工具、执行动作，就会出现一个很现实的问题：**谁来管它？**

比如客服 Agent 可以查订单、查规则、创建工单。听起来很顺，但真实系统里还要考虑：调用次数别失控、工具失败要兜底、敏感信息别乱传、关键操作要有人确认。

Middleware 做的就是这件事：**在 Agent 执行过程中加一层控制。**

## 一、Middleware 插在哪里

一个典型 Agent 循环大概是这样：

```text
用户问题 -> 调模型 -> 模型决定调工具 -> 执行工具 -> 观察结果 -> 再调模型 -> 最终回答
```

Middleware 可以插在这些关键位置：

```text
before_agent   Agent 开始前
before_model   调模型前
wrap_model     包住模型调用
after_model    模型返回后
wrap_tool      包住工具调用
after_agent    Agent 结束后
```

换句话说，它不是替 Agent 做业务，而是在 Agent 做业务时负责看门、记账、限速、兜底。没有 Middleware 的 Agent，有点像刚拿到权限的新同事：热情是有的，边界感还得配。

## 二、先准备一个普通 Agent

这里还是用售后场景。用户问订单能不能退款，Agent 可以查订单状态，也可以创建跟进工单。

先看一个没有 Middleware 的 Agent。

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool

load_dotenv(dotenv_path=".env")

# 模型需要支持 tool calling，否则 Agent 无法稳定选择和调用工具。
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    base_url=os.environ.get("DASHSCOPE_BASE_URL"),
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
)

ORDERS = {
    "O-1001": {"item": "蓝牙耳机", "status": "未发货"},
    "O-1002": {"item": "机械键盘", "status": "已发货"},
}


@tool
def get_order_status(order_id: str) -> dict:
    """根据订单号查询订单状态。"""
    # 真实项目里这里通常会查询订单系统。
    return {"order_id": order_id, **ORDERS.get(order_id, {})}


@tool
def create_refund_ticket(order_id: str, reason: str) -> dict:
    """创建退款跟进工单。"""
    # 真实项目里这里可能会写入 CRM 或工单系统。
    return {"ticket_id": "TK-0001", "order_id": order_id, "reason": reason}


tools = [get_order_status, create_refund_ticket]


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=(
        "你是一个电商售后 Agent。"
        "回答退款问题前，必须先查询订单状态。"
        "只有用户明确要求登记、跟进或创建工单时，才创建退款工单。"
    ),
)


这个 Agent 已经能工作，但它缺少一层“运行时管理”。

比如你现在看不到它调模型前带了多少消息，也看不到它准备调用哪个工具。如果工具失败、调用过多、涉及敏感信息，也没有统一入口处理。业务代码越写越多，最后就会变成一锅小补丁。

## 三、加一层 Middleware 看执行过程

下面加两个很轻的 Middleware：

- `before_model`：模型调用前打印消息数量
- `wrap_tool_call`：工具调用前后打印日志

这不是为了做复杂功能，只是先看清楚 Middleware 插在什么位置。

In [ ]:
from langchain.agents.middleware import before_model, wrap_tool_call, AgentState


@before_model
def log_before_model(state: AgentState, runtime):
    """模型调用前执行。"""
    # 可以在这里做日志、限流、上下文检查等事情。
    print(f"[middleware] 即将调用模型，当前 messages 数量：{len(state['messages'])}")
    return None


@wrap_tool_call
def log_tool_call(request, handler):
    """工具调用外层执行。"""
    tool_name = request.tool_call["name"]
    print(f"[middleware] 准备调用工具：{tool_name}")

    # handler(request) 才是真正执行工具的地方。
    result = handler(request)

    print(f"[middleware] 工具调用完成：{tool_name}")
    return result


In [ ]:
agent_with_middleware = create_agent(
    model=model,
    tools=tools,
    # 在这里配置当前 Agent 要启用哪些 Middleware。
    # Agent 执行时，会按这些 Middleware 对模型调用、工具调用等过程进行拦截和处理。
    middleware=[
        log_before_model,
        log_tool_call,
    ],
    system_prompt=(
        "你是一个电商售后 Agent。"
        "回答退款问题前，必须先查询订单状态。"
        "只有用户明确要求登记、跟进或创建工单时，才创建退款工单。"
    ),
)


现在再调用这个 Agent，业务能力没有变，但执行过程多了一层可观察入口。

In [ ]:
response = agent_with_middleware.invoke({
    "messages": [
        {
            "role": "user",
            "content": "订单 O-1001 想退款，请先判断状态，并创建一个跟进工单。",
        }
    ]
})

print(response["messages"][-1].content)


运行后你会看到类似如上的日志，这就是 Middleware 的价值：它不改变“售后 Agent 是干什么的”，但能控制和观察“Agent 是怎么干的”。这个边界很重要。

## 四、Middleware 和 Tool、Prompt 有什么区别

三者很容易混在一起，但职责完全不同。

```text
Tool：给 Agent 一个可执行能力
Prompt：告诉 Agent 应该怎么工作
Middleware：控制 Agent 工作过程
```

放到售后场景里：

- `get_order_status` 是 Tool，因为它真的去查订单。
- “必须先查订单状态”是 Prompt，因为它在约束模型行为。
- “调用工具前打印日志、失败时重试、敏感信息脱敏”是 Middleware，因为它管理运行过程。

如果 Tool 是员工手里的工具箱，Prompt 是工作说明书，那 Middleware 就是门禁、摄像头、审批流和保险丝。没有它也能跑，但你不一定睡得着。

## 五、官方常见 Middleware 速查

LangChain v1 已经内置了一批 Middleware。先不用急着全学会，知道它们各自解决什么问题就够了。

| Middleware | 解决什么问题 | 典型场景 |
|---|---|---|
| `SummarizationMiddleware` | 对话太长时自动总结历史消息 | 客服连续聊了几十轮，不能每次都把完整聊天记录塞给模型 |
| `HumanInTheLoopMiddleware` | 关键工具调用前暂停，等待人工审批 | 退款、发邮件、删数据、提交审批这类高风险动作 |
| `ModelCallLimitMiddleware` | 限制模型调用次数 | 防止 Agent 反复思考，API 账单开始偷偷加班 |
| `ToolCallLimitMiddleware` | 限制工具调用次数 | 限制搜索、查库、爬网页等外部调用，避免跑飞 |
| `ModelFallbackMiddleware` | 主模型失败时切换备用模型 | 主模型超时、服务不可用时，用备用模型继续完成任务 |
| `ModelRetryMiddleware` | 模型调用失败后自动重试 | 模型接口偶发超时、网络抖动 |
| `ToolRetryMiddleware` | 工具调用失败后自动重试 | 订单 API、库存 API、搜索 API 临时失败 |
| `PIIMiddleware` | 识别并处理敏感信息 | 手机号、邮箱、信用卡号进入模型前先脱敏 |
| `ContextEditingMiddleware` | 修剪或清理上下文 | 工具结果太多、历史消息太乱，需要控制上下文体积 |
| `LLMToolSelectorMiddleware` | 先筛选相关工具，再交给主模型 | 工具很多时，先缩小候选范围，减少模型选择压力 |
| `LLMToolEmulator` | 用 LLM 模拟工具执行 | 工具还没接好时，先验证 Agent 流程 |
| `TodoListMiddleware` | 给 Agent 增加任务规划和跟踪能力 | 多步骤任务，比如整理资料、生成报告、逐项检查 |
| `FilesystemMiddleware` | 给 Agent 提供文件系统能力 | 让 Agent 保存中间结果、读取文件、维护长期上下文 |
| `ShellToolMiddleware` | 给 Agent 暴露 shell 执行能力 | 需要执行命令的开发类 Agent；权限要管好，不然刺激过头 |
| `SubAgentMiddleware` | 让 Agent 可以调用子 Agent | 一个主 Agent 分派任务给搜索、分析、写作等专业 Agent |

这张表不需要一次背下来。更实用的理解方式是：

- **怕失控**：看调用限制类 Middleware。
- **怕失败**：看 retry / fallback。
- **怕误操作**：看 human-in-the-loop。
- **怕泄露和上下文爆炸**：看 PII、summarization、context editing。
- **想扩展复杂能力**：看 filesystem、subagent、tool selector。